# Conjunction Hackathon — Student Tutorial

This notebook walks you through the toolkit **step by step**: load a satellite catalog, propagate orbits, visualize trajectories, screen for close approaches (conjunctions), and verify a claim.

**Run cells in order.** Each section builds on the previous one.

### What you will learn

1. Load and inspect orbital catalogs (SpaceTrack JSON / TLE)
2. Turn elements into Skyfield/SGP4 satellites
3. Propagate positions and plot orbits
4. Measure miss distance and time of closest approach (TCA)
5. Screen many pairs (baseline vs fast)
6. Submit and verify a `ConjunctionClaim`

## 0. Setup

**Google Colab**
1. Open [colab.research.google.com](https://colab.research.google.com/) and **Sign in**
2. **Upload notebook** → choose `conjunction_tutorial.ipynb`
3. Run the next cells: install packages, then fetch data (git clone **or** upload `colab_conjunction_bundle.zip`)

**Local**
Activate the project `.venv`, open this notebook from the repo root, and skip install/clone if already set up.


In [ ]:
# Colab / fresh environment: install dependencies
import sys
import subprocess

pkgs = [
    "skyfield>=1.48",
    "sgp4>=2.23",
    "numpy>=1.26",
    "scipy>=1.11",
    "plotly>=5.18",
    "pandas>=2.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencies installed.")


In [ ]:
# Fetch toolkit + spacetrack_data.json into the working directory.
# Order: already present → clone GitHub → (Colab) upload colab_conjunction_bundle.zip
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/abensour/collision_detection_hackathon.git"
REPO_BRANCH = "main"
CLONE_DIR = Path("/content/collision_detection_hackathon")
BUNDLE_NAME = "colab_conjunction_bundle.zip"


def find_project_root():
    candidates = [
        Path.cwd(),
        CLONE_DIR,
        Path.cwd() / "collision_detection_hackathon",
        Path.cwd() / "conjections_hackaton",
        Path.cwd() / "colab_bundle",
        Path("/content/colab_bundle"),
    ]
    for p in candidates:
        if (p / "conjunction_toolkit").exists() and (p / "spacetrack_data.json").exists():
            return p.resolve()
    return None


def looks_like_colab() -> bool:
    return "google.colab" in sys.modules or Path("/content").exists()


ROOT = find_project_root()

if ROOT is None and looks_like_colab():
    try:
        print(f"Cloning {REPO_URL} ({REPO_BRANCH}) …")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)]
        )
        ROOT = find_project_root()
    except Exception as exc:
        print("Git clone failed:", exc)

if ROOT is None and looks_like_colab():
    from google.colab import files

    print(
        f"Upload {BUNDLE_NAME} from your project folder "
        "(contains toolkit + spacetrack_data.json)."
    )
    uploaded = files.upload()
    zip_path = None
    for name in uploaded:
        if name.endswith(".zip"):
            zip_path = Path(name)
            break
    if zip_path is None:
        raise FileNotFoundError(f"Please upload {BUNDLE_NAME}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    ROOT = find_project_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not find conjunction_toolkit/ + spacetrack_data.json. "
        "Locally: open the notebook from the project root. "
        "On Colab: allow git clone, or upload colab_conjunction_bundle.zip."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Catalog present:", (ROOT / "spacetrack_data.json").exists())


In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "conjunction_toolkit").exists():
    raise FileNotFoundError(
        "conjunction_toolkit/ not found. Run the setup cells above first "
        "(Colab clone / local project root)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from conjunction_toolkit import (
    ConjunctionClaim,
    VerifyConfig,
    catalog_to_satellites,
    closest_approach_on_grid,
    load_default_catalog,
    pair_distances,
    plot_pair_with_distance,
    plot_trajectories,
    refine_closest_approach,
    save_html,
    screen_pairs,
    screen_pairs_fast,
    time_grid,
    verify_claim,
)

OUTPUT_DIR = ROOT / "examples" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. Project root:", ROOT)
print("Catalog present:", (ROOT / "spacetrack_data.json").exists())

Ready. Project root: /Users/lishayabensour/conjections_hackaton


## 1. Satellite basics (quick theory)

A satellite stays in orbit when its sideways speed matches Earth’s gravity at that altitude. For a **circular** orbit (two-body):

$$
v = \sqrt{\frac{GM}{r}}
\qquad
T = 2\pi\sqrt{\frac{r^{3}}{GM}}
$$

| Symbol | Meaning |
|--------|---------|
| $v$ | Orbital speed |
| $T$ | Orbital period |
| $r$ | Distance from Earth’s **center** (not height above the surface) |
| $GM$ | ≈ $3.986 \times 10^{5}\,\mathrm{km}^{3}/\mathrm{s}^{2}$ |

Height above the surface is $h = r - R_E$ with $R_E \approx 6371\,\mathrm{km}$. Higher orbits are slower.

| Regime | Approx. altitude | Rough period |
|--------|------------------|--------------|
| LEO | 200–2000 km | ~90–130 min |
| MEO | ~20 000 km | ~12 h |
| GEO | ~35 786 km | ~24 h |

Most catalog objects here are **LEO**: they move fast, and relative geometry between two objects can change quickly.

### Conjunction vocabulary

| Term | Meaning |
|------|---------|
| **Conjunction** | Two objects coming close in space (near the same time) |
| **TCA** | Time of closest approach |
| **Miss distance** | Minimum separation at TCA |

Positions in this toolkit are **GCRS**, in **kilometers**.

## 2. Load the catalog

`load_default_catalog()` reads `spacetrack_data.json` when present.  
On hackathon day you may instead use `load_tle_file("catalog.tle")` — the rest of the API stays the same.

In [ ]:
catalog = load_default_catalog()

print(f"Source:  {catalog.source_path}")
print(f"Objects: {len(catalog)}")

epochs = [el.epoch_utc for el in catalog]
print(f"Epoch range: {min(epochs).isoformat()} → {max(epochs).isoformat()}")

## 3. Inspect orbital elements

Each object has classical elements: inclination, mean motion, eccentricity, epoch, etc.

In [ ]:
print("Sample objects:")
for nid in catalog.ids()[:5]:
    el = catalog[nid]
    print(
        f"  NORAD {el.norad_cat_id:>6}  {el.object_name:<24}  "
        f"i={el.inclination_deg:6.2f}°  "
        f"n={el.mean_motion_rev_per_day:.6f} rev/day  "
        f"epoch={el.epoch_utc.isoformat()}"
    )

iss = catalog.filter_by_name("ISS (ZARYA)")
print(f"\nExact name 'ISS (ZARYA)': {len(iss)} object(s)")
for el in iss:
    print(
        f"  NORAD {el.norad_cat_id}: {el.object_name} "
        f"epoch={el.epoch_utc.isoformat()}"
    )

### Try it
Change the name substring below (e.g. `"STARLINK"`, `"HST"`, `"ONEWEB"`) and see how many objects match.

In [ ]:
name_query = "STARLINK"  # <-- edit me
hits = catalog.filter_by_name(name_query)
print(f"Objects matching '{name_query}': {len(hits)}")
for el in list(hits)[:5]:
    print(f"  {el.norad_cat_id}  {el.object_name}")

## 4. Build satellites and propagate

Flow:

```
Catalog / OrbitalElements  →  EarthSatellite (SGP4)  →  positions over a time grid
```

We pick a few well-known objects (or fall back to recent LEO), then propagate for a few hours.

In [ ]:
# Prefer familiar names; fall back to recent LEO if needed
preferred: list[int] = []
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    hits = [
        el
        for el in catalog.filter_by_name(name)
        if el.object_name == name or el.object_name.startswith(name + " ")
    ]
    if not hits:
        hits = list(catalog.filter_by_name(name))
    if hits:
        hits.sort(key=lambda el: el.epoch_utc, reverse=True)
        preferred.append(hits[0].norad_cat_id)

if len(preferred) < 2:
    leo = sorted(
        (el for el in catalog if el.mean_motion_rev_per_day > 14),
        key=lambda el: el.epoch_utc,
        reverse=True,
    )
    preferred = [el.norad_cat_id for el in leo[:3]]

preferred = preferred[:3]
sats = catalog_to_satellites(catalog, norad_ids=preferred)

epochs = [catalog[nid].epoch_utc for nid in preferred]
t0 = max(epochs)
t1 = t0 + timedelta(hours=6)
t = time_grid(t0, t1, step_seconds=60.0)

print("Selected:", [(nid, sats[nid].name) for nid in preferred])
print(f"Window:  {t0.isoformat()} → {t1.isoformat()}")
print(f"Time steps: {len(t)}")

## 5. Visualize trajectories

Plotly writes an interactive HTML file you can open in a browser.

In [ ]:
fig = plot_trajectories(sats, t, title="Sample trajectories (GCRS)")
traj_path = OUTPUT_DIR / "notebook_trajectories.html"
save_html(fig, traj_path)
print(f"Wrote {traj_path}")

# Show inline in the notebook as well
fig.show()

## 6. Pair distance and closest approach on a grid

For two satellites, compute distance at every timestep, then find the coarse TCA on that grid.  
`refine_closest_approach` then zooms in for a better TCA / miss distance.

In [ ]:
id_a, id_b = preferred[0], preferred[1]
sat_a, sat_b = sats[id_a], sats[id_b]

distances = pair_distances(sat_a, sat_b, t)
print(f"Pair {id_a}–{id_b}")
print(f"  min distance on grid: {distances.min():.3f} km")
print(f"  max distance on grid: {distances.max():.3f} km")

tca_grid, dmin_grid = closest_approach_on_grid(sat_a, sat_b, t)
print(f"  grid TCA: {tca_grid.isoformat()}  miss={dmin_grid:.3f} km")

tca_ref, dmin_ref = refine_closest_approach(sat_a, sat_b, tca_grid)
print(f"  refined TCA: {tca_ref.isoformat()}  miss={dmin_ref:.3f} km")

In [ ]:
# Zoom around the refined TCA for a clearer close-approach plot
t_zoom = time_grid(
    tca_ref - timedelta(minutes=30),
    tca_ref + timedelta(minutes=30),
    step_seconds=30.0,
)
fig_pair = plot_pair_with_distance(
    sat_a,
    sat_b,
    t_zoom,
    tca=tca_ref,
    norad_a=id_a,
    norad_b=id_b,
    threshold_km=100.0,
    title=f"Close approach {id_a}–{id_b}",
)
pair_path = OUTPUT_DIR / "notebook_pair.html"
save_html(fig_pair, pair_path)
print(f"Wrote {pair_path}")
fig_pair.show()

## 7. Baseline screener (brute-force)

`screen_pairs` checks **all unique pairs** on a time grid.  
Use a **small** subset first — $N$ objects means $N(N-1)/2$ pairs.

We take a recent Starlink slice (dense constellation → more likely to show close approaches).

In [ ]:
min_epoch = datetime(2026, 1, 1, tzinfo=timezone.utc)
pool = [
    el
    for el in catalog
    if el.epoch_utc >= min_epoch and "STARLINK" in el.object_name.upper()
]
pool.sort(key=lambda el: el.norad_cat_id)

N_BASELINE = 40  # keep small for a quick notebook run; try 50–100 if you have time
subset = pool[:N_BASELINE]
screen_ids = [el.norad_cat_id for el in subset]
screen_sats = catalog_to_satellites(catalog, norad_ids=screen_ids)

screen_epochs = sorted(el.epoch_utc for el in subset)
screen_t0 = screen_epochs[len(screen_epochs) // 2]
screen_t1 = screen_t0 + timedelta(hours=6)
threshold_km = 100.0

print(f"Screening {len(screen_ids)} objects (baseline)...")
print(f"Window: {screen_t0.isoformat()} → {screen_t1.isoformat()}")
print(f"Pairs to check: {len(screen_ids) * (len(screen_ids) - 1) // 2}")

t_start = time.perf_counter()
raw_claims = screen_pairs(
    screen_sats,
    screen_t0,
    screen_t1,
    step_seconds=60.0,
    threshold_km=threshold_km,
)
baseline_s = time.perf_counter() - t_start
print(f"Raw grid claims within {threshold_km} km: {len(raw_claims)}  ({baseline_s:.2f}s)")

In [ ]:
# Refine TCA for each hit, then verify (organizer-style check)
config = VerifyConfig(max_miss_distance_km=threshold_km)
claims: list[ConjunctionClaim] = []

for raw in raw_claims:
    tca, dmin = refine_closest_approach(
        screen_sats[raw.norad_a], screen_sats[raw.norad_b], raw.tca_utc
    )
    claims.append(
        ConjunctionClaim(
            norad_a=raw.norad_a,
            norad_b=raw.norad_b,
            tca_utc=tca,
            min_distance_km=dmin,
            algorithm_id="baseline_bruteforce_refined",
        )
    )

claims.sort(key=lambda c: c.min_distance_km)
print(f"Refined claims: {len(claims)}")
print("\nTop hits (verified):")
for claim in claims[:10]:
    result = verify_claim(claim, screen_sats, config=config)
    status = "OK" if result.ok else "FAIL"
    print(
        f"  [{status}] {claim.norad_a}–{claim.norad_b}  "
        f"claimed {claim.min_distance_km:.3f} km @ {claim.tca_utc.isoformat()}  "
        f"→ recomputed {result.recomputed_distance_km:.3f} km"
    )

## 8. Fast screener

`screen_pairs_fast` is much cheaper for larger $N$:

1. **Altitude bands** — skip pairs that cannot meet within `threshold + pad`
2. **Spatial KD-tree** each timestep — only neighbor pairs within `threshold_km`
3. Full-grid miss distance on candidates, then optional **TCA refine**

Compare it to the baseline on the same subset.

In [ ]:
fast_threshold_km = 100.0  # same window/sats as baseline section

t_a = time.perf_counter()
baseline = screen_pairs(
    screen_sats,
    screen_t0,
    screen_t1,
    step_seconds=60.0,
    threshold_km=fast_threshold_km,
)
baseline_s = time.perf_counter() - t_a

t_b = time.perf_counter()
fast_grid = screen_pairs_fast(
    screen_sats,
    screen_t0,
    screen_t1,
    step_seconds=60.0,
    threshold_km=fast_threshold_km,
    refine=False,
    algorithm_id="spatial_hash_grid",
)
fast_grid_s = time.perf_counter() - t_b

t_c = time.perf_counter()
fast = screen_pairs_fast(
    screen_sats,
    screen_t0,
    screen_t1,
    step_seconds=60.0,
    threshold_km=fast_threshold_km,
    refine=True,
    algorithm_id="spatial_hash",
)
fast_s = time.perf_counter() - t_c


def pair_set(claims):
    return {(c.norad_a, c.norad_b) for c in claims}


base_pairs = pair_set(baseline)
fast_pairs = pair_set(fast_grid)
missing = base_pairs - fast_pairs
extra = fast_pairs - base_pairs

print(f"Baseline brute-force: {len(baseline)} claims in {baseline_s:.3f}s")
print(f"Fast (grid only):     {len(fast_grid)} claims in {fast_grid_s:.3f}s")
print(f"Fast (with refine):   {len(fast)} claims in {fast_s:.3f}s")
if baseline_s > 0:
    print(f"Speedup (grid vs baseline): {baseline_s / max(fast_grid_s, 1e-9):.1f}x")
print(f"\nPair-set match (grid): missing={len(missing)} extra={len(extra)}")
if missing:
    print(f"  missing examples: {list(missing)[:5]}")
if extra:
    print(f"  extra examples: {list(extra)[:5]}")

## 9. Claiming and verifying a conjunction

A claim is: two NORAD IDs + TCA + miss distance.

`verify_claim()` re-propagates both objects and accepts only if distance and TCA match within tolerances (default: **100 m**, **5 s**).

**Tip:** refine your TCA before submitting — a coarse grid alone often fails those tolerances.

In [ ]:
# Demo: honest claim should PASS; fake distance should FAIL
verify_config = VerifyConfig(
    distance_tolerance_km=0.1,   # 100 m
    tca_tolerance_seconds=5.0,
    max_miss_distance_km=None,   # only check claim consistency here
)

if claims:
    best = claims[0]
    honest = ConjunctionClaim(
        norad_a=best.norad_a,
        norad_b=best.norad_b,
        tca_utc=best.tca_utc,
        min_distance_km=best.min_distance_km,
        algorithm_id="notebook_honest",
    )
else:
    # Fallback if baseline found nothing: refine any two LEO objects
    leo = [
        el
        for el in catalog
        if el.mean_motion_rev_per_day > 14.0 and el.eccentricity < 0.02
    ][:30]
    demo_sats = catalog_to_satellites(catalog, norad_ids=[el.norad_cat_id for el in leo])
    a_id, b_id = leo[0].norad_cat_id, leo[1].norad_cat_id
    approx = max(leo[0].epoch_utc, leo[1].epoch_utc)
    tca, dmin = refine_closest_approach(demo_sats[a_id], demo_sats[b_id], approx)
    screen_sats = demo_sats
    honest = ConjunctionClaim(
        norad_a=a_id,
        norad_b=b_id,
        tca_utc=tca,
        min_distance_km=dmin,
        algorithm_id="notebook_honest",
    )

print(
    f"Honest claim: {honest.norad_a}–{honest.norad_b}  "
    f"{honest.min_distance_km:.6f} km @ {honest.tca_utc.isoformat()}"
)
result = verify_claim(honest, screen_sats, config=verify_config)
print(f"  ok={result.ok}")
print(f"  recomputed_distance_km={result.recomputed_distance_km:.6f}")
print(f"  distance_error_km={result.distance_error_km:.6f}")
print(f"  tca_error_seconds={result.tca_error_seconds}")
for msg in result.messages:
    print(f"  - {msg}")

fake = ConjunctionClaim(
    norad_a=honest.norad_a,
    norad_b=honest.norad_b,
    tca_utc=honest.tca_utc,
    min_distance_km=0.001,  # deliberately wrong
    algorithm_id="notebook_fake",
)
fake_result = verify_claim(fake, screen_sats, config=verify_config)
print("\nFake claim (should FAIL):")
print(f"  ok={fake_result.ok}")
for msg in fake_result.messages:
    print(f"  - {msg}")

## 10. Your challenge

Use the cells below (or add your own) to:

1. Choose a subset of the catalog (by name, epoch, LEO cut, …)
2. Screen for conjunctions with `screen_pairs` or `screen_pairs_fast`
3. Refine TCA for your best hits
4. Build a `ConjunctionClaim` and run `verify_claim`

Acceptance defaults: distance within **0.1 km**, TCA within **5 s**.

In [ ]:
# --- Student workspace ---
# Edit these knobs, then run.

MY_N = 80
MY_THRESHOLD_KM = 25.0
MY_HOURS = 6
USE_FAST = True  # False → baseline screen_pairs

my_pool = [
    el
    for el in catalog
    if el.epoch_utc >= min_epoch and "STARLINK" in el.object_name.upper()
]
my_pool.sort(key=lambda el: el.norad_cat_id)
my_subset = my_pool[:MY_N]
my_ids = [el.norad_cat_id for el in my_subset]
my_sats = catalog_to_satellites(catalog, norad_ids=my_ids)

my_epochs = sorted(el.epoch_utc for el in my_subset)
my_t0 = my_epochs[len(my_epochs) // 2]
my_t1 = my_t0 + timedelta(hours=MY_HOURS)

print(f"Objects={len(my_ids)}  threshold={MY_THRESHOLD_KM} km")
print(f"Window: {my_t0.isoformat()} → {my_t1.isoformat()}")

t0_run = time.perf_counter()
if USE_FAST:
    my_hits = screen_pairs_fast(
        my_sats,
        my_t0,
        my_t1,
        step_seconds=60.0,
        threshold_km=MY_THRESHOLD_KM,
        refine=True,
        algorithm_id="student_fast",
    )
else:
    raw = screen_pairs(
        my_sats,
        my_t0,
        my_t1,
        step_seconds=60.0,
        threshold_km=MY_THRESHOLD_KM,
    )
    my_hits = []
    for r in raw:
        tca, dmin = refine_closest_approach(my_sats[r.norad_a], my_sats[r.norad_b], r.tca_utc)
        my_hits.append(
            ConjunctionClaim(
                norad_a=r.norad_a,
                norad_b=r.norad_b,
                tca_utc=tca,
                min_distance_km=dmin,
                algorithm_id="student_baseline",
            )
        )
elapsed = time.perf_counter() - t0_run

my_hits = sorted(my_hits, key=lambda c: c.min_distance_km)
print(f"Found {len(my_hits)} claims in {elapsed:.2f}s")

my_config = VerifyConfig(max_miss_distance_km=MY_THRESHOLD_KM)
for claim in my_hits[:5]:
    result = verify_claim(claim, my_sats, config=my_config)
    status = "OK" if result.ok else "FAIL"
    print(
        f"  [{status}] {claim.norad_a}–{claim.norad_b}  "
        f"{claim.min_distance_km:.3f} km @ {claim.tca_utc.isoformat()}"
    )

if my_hits:
    best = my_hits[0]
    print("\nBest claim to submit:")
    print(f"  norad_a = {best.norad_a}")
    print(f"  norad_b = {best.norad_b}")
    print(f"  tca_utc = {best.tca_utc.isoformat()}")
    print(f"  min_distance_km = {best.min_distance_km:.6f}")

---

### Cheat sheet

| Goal | Function |
|------|----------|
| Load catalog | `load_default_catalog()` / `load_tle_file()` |
| Build sats | `catalog_to_satellites(catalog, norad_ids=...)` |
| Time grid | `time_grid(t0, t1, step_seconds=60)` |
| Plot orbits | `plot_trajectories(sats, t)` |
| Pair miss on grid | `closest_approach_on_grid(a, b, t)` |
| Refine TCA | `refine_closest_approach(a, b, approx_tca)` |
| Brute-force screen | `screen_pairs(...)` |
| Fast screen | `screen_pairs_fast(..., refine=True)` |
| Check a claim | `verify_claim(claim, sats, config=...)` |

Typical data flow:

```
catalog file → parse → satellites → propagate → visualize / screen / verify
```